In [1]:
import json
import os
import sys
import time
from pathlib import Path

# **Primo test Baseline**


In [2]:
#(LOCAL vs COLAB)
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    print("🌍 Detetected enviroment: COLAB")
    from google.colab import drive
    drive.mount('/content/drive')
    input_dir = Path('')
    output_dir = Path('')
else:
    print("💻 Detetected enviroment: LOCAL")
    # Impostate la vostra cartella di input e output locale
    input_dir = Path('../data/')
    output_dir = Path('../data/output')

    # Crea le cartelle se non esistono in locale
    output_dir.mkdir(parents=True, exist_ok=True)
    
INPUT_DIR, OUTPUT_DIR = input_dir, output_dir
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

💻 Detetected enviroment: LOCAL
Input directory: ../data
Output directory: ../data/output


## API milionario


In [3]:
#OCCHIO CHE VB AGGIUNGE SYS PATH QUINDI METTETE PERCORSO GIUSTO (teoricamente è giusto già)

# Define the path to the directory containing your package
package_parent_dir = './'
package_parent_dir = os.path.abspath(package_parent_dir)
#print(package_parent_dir)

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)
    

# Verify the path was added
#print(sys.path)


#import your package
from millionaire_client import MillionaireClient, AuthenticationError
import dotenv
dotenv.load_dotenv()

#credentiali per accedere al server
username = os.getenv('USER')
pwd = os.getenv('PASSWORD')
mail=os.getenv('MAIL')
API_URL = "http://131.175.15.22:51111/"

# Create a client instance
client = MillionaireClient(API_URL)
print(f"username: {username}, password: {pwd}, mail: {mail}, API_URL: {API_URL}")
try:
    # Authenticate with the server
    user = client.login(username, pwd)
    print("✅ Login successful :)")
    print(f"\n Welcome, {user.username}! Role: {user.role}")
except AuthenticationError as e:
    print(f"❌ Login failed: {e}")

username: lillaro, password: lillarowins01, mail: /var/spool/mail/lillaro, API_URL: http://131.175.15.22:51111/
✅ Login successful :)

 Welcome, lillaro! Role: student


## Local LLM Integration
Random small models
Install the extra requirements first:
  pip install -r requirements.txt

In [4]:
from transformers import pipeline
import torch

print("Loading local language model... (this may take a moment)")
llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen3-VL-2B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",  # GPU if available, otherwise CPU
    trust_remote_code=False,
)
print("Model loaded successfully!")

/home/lillaro/Documents/Programmazione/NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading local language model... (this may take a moment)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 625/625 [00:00<00:00, 795.41it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
[transformers] The model 'Qwen3VLForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausa

Model loaded successfully!


In [8]:
# Model configuration and data persistence setup
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
REASONING = False  # Set to True for reasoning "on", False for "off"

# Create data directory for results
DATA_DIR = Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)


def sanitize_filename(name: str) -> str:
    """Replace filesystem-unfriendly characters in model names."""
    return name.replace('/', '-').replace('\\', '-')


def get_json_path(model_name: str, reasoning: bool) -> Path:
    """
    Determine the correct JSON file path.
    If a file with the same model name exists but different reasoning setting,
    increment suffix (e.g. model-2.json, model-3.json, ...).
    """
    base_name = sanitize_filename(model_name)
    suffix = 1

    while True:
        if suffix == 1:
            filename = f"{base_name}.json"
        else:
            filename = f"{base_name}-{suffix}.json"

        filepath = DATA_DIR / filename

        if not filepath.exists():
            return filepath

        # File exists: check if settings match
        with open(filepath, 'r', encoding='utf-8') as f:
            existing = json.load(f)

        if existing.get('model_name') == model_name and existing.get('reasoning') == reasoning:
            return filepath

        # Settings differ, try next suffix
        suffix += 1


def save_result(filepath: Path, model_name: str, reasoning: bool, competition_id, earned_amount: float, max_time: float):
    """Append a game result to the JSON file, keyed by competition ID."""

    if filepath.exists():
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        data = {
            "model_name": model_name,
            "reasoning": reasoning,
            "data": {}
        }

    comp_key = str(competition_id)
    if comp_key not in data["data"]:
        data["data"][comp_key] = []

    data["data"][comp_key].append({
        "earned": earned_amount,
        "time": max_time
    })

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)

    print(f"Saved result to {filepath}")

In [9]:
class LLMStrategy:
    """
    Callable strategy that asks the local LLM to pick the best option.
    Tracks the maximum time spent on any single question during a game.
    """
    def __init__(self, enable_thinking=False):
        self.enable_thinking = enable_thinking
        self.max_time = 0.0

    def __call__(self, question):
        if not question or not question.options:
            raise ValueError("No question/options provided")

        start = time.time()

        letters = ['A', 'B', 'C', 'D']
        opts = question.options[:4]  # never more than 4 options
        options_text = "\n".join(
            [f"{letters[i]}: {opt.text}" for i, opt in enumerate(opts)]
        )

        prompt = (
            f"Question: {question.text}\n"
            f"Options:\n{options_text}\n"
            f"Answer with just the letter of the correct option (A, B, C, or D).\n"
            f"Answer:"
        )

        # Format as a chat so the instruct model behaves correctly
        messages = [{"role": "user", "content": prompt}]
        try:
            prompt_text = llm_pipeline.tokenizer.apply_chat_template(
                messages, tokenize=False, enable_thinking=self.enable_thinking
            )
        except Exception:
            prompt_text = prompt  # Fallback for non-chat tokenizers

        outputs = llm_pipeline(
            prompt_text,
            max_new_tokens=5,
            do_sample=False,
            return_full_text=False,
        )

        generated = outputs[0]["generated_text"].strip().upper()
        print(f"Prompt: {prompt_text}")
        print(f"[LLM raw output] '{generated}'")

        elapsed = time.time() - start
        if elapsed > self.max_time:
            self.max_time = elapsed

        for char in generated:
            if char in letters:
                idx = letters.index(char)
                return opts[idx].id

        print("Could not parse a valid letter from LLM output, falling back to first option.")
        return opts[0].id

In [10]:
%%time
# 1. List all competitions and determine JSON target
comps = client.competitions.list_all()
print(f"Found {len(comps)} competitions: {comps}")

json_path = get_json_path(MODEL_NAME, REASONING)
print(f"Using results file: {json_path}")

# 2. Play one game per competition ID using the LLM
for comp in comps:
    comp_id = comp.id
    print(f"\n--- Starting run for competition ID {comp_id} ---")

    strategy = LLMStrategy(enable_thinking=REASONING)
    final = client.play_game(comp_id, strategy)
    print(f"Game over! Earned amount: {final.earned_amount}, max question time: {strategy.max_time:.3f}s")

    save_result(json_path, MODEL_NAME, REASONING, comp_id, final.earned_amount, strategy.max_time)

print("\nAll competitions completed.")

[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Found 4 competitions: [Competition(id=0, name='Entertainment', description='Music, Movies, Celebrities and more', max_levels=15, is_infinite=False, created_at='2026-04-12 19:29:31', question_count=843), Competition(id=1, name='Ancient History and Politics', description='The Roman Empire, The Greeks, and more', max_levels=15, is_infinite=False, created_at='2026-04-12 19:29:57', question_count=454), Competition(id=2, name='Science and Nature', description='Chemistry, Biology, Physics and similar subjects', max_levels=15, is_infinite=False, created_at='2026-04-12 19:30:13', question_count=5395), Competition(id=3, name='Maths', description='Mathematics and Statistics from High School and College', max_levels=15, is_infinite=False, created_at='2026-04-12 19:30:22', question_count=390)]
Using results file: data/Qwen-Qwen3-VL-2B-Instruct.json

--- Starting run for competition ID 0 ---


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: <|im_start|>user
Question: How did Streep's relationship with her mother influence her career?
Options:
A: Her mother's confidence in her boosted Streep's self-esteem
B: Her mother discouraged her from pursuing acting
C: Her mother was a famous actress who mentored her
D: Her mother taught her how to act
Answer with just the letter of the correct option (A, B, C, or D).
Answer:<|im_end|>

[LLM raw output] 'A'


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Prompt: <|im_start|>user
Question: What is the primary reason quantum mechanics is necessary to describe systems at very small scales?
Options:
A: It allows for the precise prediction of particle positions.
B: It simplifies the description of macroscopic phenomena.
C: It eliminates the need for understanding black-body radiation.
D: It provides a framework for understanding the wave-particle duality of matter and light.
Answer with just the letter of the correct option (A, B, C, or D).
Answer:<|im_end|>

[LLM raw output] 'D'


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: <|im_start|>user
Question: What is the fundamental principle of total internal reflection?
Options:
A: The refracted ray is always parallel to the normal.
B: The angle of incidence exceeds the critical angle, leading to total reflection.
C: The incident ray is always perpendicular to the surface.
D: The angle of incidence is always 90 degrees.
Answer with just the letter of the correct option (A, B, C, or D).
Answer:<|im_end|>

[LLM raw output] 'B'


[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: <|im_start|>user
Question: Students made lemonade using the following recipe: 100 grams of lemon juice 100 grams of sugar 1,000 grams of water The students combined the lemon juice, sugar, and water in a container. They stirred the lemonade until all the sugar dissolved. They poured the lemonade into a plastic tray and put the tray in a freezer. The next day, the students removed the tray from the freezer and observed that the lemonade was a solid. What is the mass of the solid lemonade?
Options:
A: 1,000 grams
B: 1,200 grams
C: 200 grams
D: 100 grams
Answer with just the letter of the correct option (A, B, C, or D).
Answer:<|im_end|>

[LLM raw output] 'C'
Game over! Earned amount: 4000, max question time: 1.538s
Saved result to data/Qwen-Qwen3-VL-2B-Instruct.json

--- Starting run for competition ID 3 ---
Prompt: <|im_start|>user
Question: Statement 1 | Every quotient ring of every commutative ring is again a commutative ring. Statement 2 | An ideal N in a ring R with unity 1 